In [ ]:
# Step 1
# Install bleeding edge dependencies transformers, PEFT to support gemma4 configurations and trl for DPO
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/trl.git
!pip install -q accelerate bitsandbytes datasets numpy wandb tensorboard

In [ ]:
# Step 2
# Imports and Configuration
import torch
import wandb
import os
from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import DPOTrainer

MODEL_ID = "google/gemma-4-e4b"
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
DPO_BETA = 0.1 # Temperature parameter for DPO
OUTPUT_DIR = "./gemma4-e4b-dpo-jokes"

torch.manual_seed(42)

In [ ]:
# Step 3
# Load Base Model in 4-bit precision
# Inject LoRA Adapters for DPO

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    quantization_config=bnb_config,
    dtype=torch.float16
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj.linear", "v_proj.linear", "k_proj.linear", "o_proj.linear"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
# Step 4
# Dataset Preparation for DPO
# DPO requires a dataset with 'prompt', 'chosen', and 'rejected' columns.

raw_dataset = load_dataset("Maximofn/short-jokes-dataset", split="train[:1000]")

def format_dpo_pairs(sample):
    # Synthetic generation of rejected responses for structural demonstration.
    # In production, replace this with a true preference dataset containing human or LLM-judged rejections.
    joke = sample.get('Joke', '')
    rejected_joke = joke[:int(len(joke)/2)] + "... wait, I forgot the punchline."

    return {
        "prompt": "System: You are a comedian. Make this funnier: Tell me a joke.\nAnswer: ",
        "chosen": joke,
        "rejected": rejected_joke
    }

dpo_dataset = raw_dataset.map(format_dpo_pairs, remove_columns=raw_dataset.column_names)
train_test_split = dpo_dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

In [ ]:
# Step 5
# Execute Direct Preference Optimization (DPO) with Quality Logging

# Initialize Weights & Biases for quality tracking

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
wandb.login()
wandb.init(project="gemma4-dpo-humor", name="dpo-run-1")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=3,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=50,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    report_to=["wandb", "tensorboard"], # Logging integration
    optim="paged_adamw_32bit",
    fp16=True,
    remove_unused_columns=False
)

dpo_trainer = DPOTrainer(
    model,
    ref_model=None, # DPOTrainer automatically manages the reference model when using PEFT
    args=training_args,
    beta=DPO_BETA,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_prompt_length=128,
    max_length=256
)

dpo_trainer.train()
wandb.finish()

In [ ]:
# Step 6
# Save the final optimized adapters
dpo_trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"DPO optimized adapters saved to {OUTPUT_DIR}")